In [1]:
%pip install -q bertopic sentence-transformers safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.2 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_corpus.csv to cleaned_corpus.csv


In [3]:
import pandas as pd

df = pd.read_csv(
    "cleaned_corpus.csv",
    keep_default_na=False,
)

required_columns = {
    "id",
    "platform",
    "text_for_analysis",
    "cleaned_text",
    "is_synthetic",
}

missing = required_columns - set(df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

assert not df.empty, "The dataset is empty."
assert df["id"].is_unique, "Duplicate record IDs found."
assert df["cleaned_text"].str.strip().ne("").all(), "Empty text found."
assert df["platform"].isin(["reddit", "linkedin"]).all()

print("Dataset validation passed.")
print(f"Total records: {len(df)}")
print("\nRecords per platform:")
print(df.groupby("platform").size())

display(
    df[["platform", "text_for_analysis", "cleaned_text"]].head(3)
)

Dataset validation passed.
Total records: 20

Records per platform:
platform
linkedin    10
reddit      10
dtype: int64


,platform,text_for_analysis,cleaned_text
0,reddit,I work at TCS with a 12lpa package. I am worri...,I work at [SERVICE_COMPANY] with a [COMPENSATI...
1,reddit,Infosys put me on the bench. I am not optimist...,[SERVICE_COMPANY] put me [LABOR_PRACTICE] benc...
2,reddit,My 90-day notice period is making it difficult...,My [LABOR_PRACTICE] notice_period is making it...


In [4]:
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

print("BERTopic import successful.")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

documents = df["cleaned_text"].head(3).tolist()

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=False,
)

assert embeddings.shape == (3, 384)
assert np.isfinite(embeddings).all()

print(f"Documents encoded: {len(documents)}")
print(f"Embedding shape: {embeddings.shape}")
print("Model setup successful.")

BERTopic import successful.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents encoded: 3
Embedding shape: (3, 384)
Model setup successful.


In [5]:
%pip install -q bertopic sentence-transformers safetensors

In [7]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload only the updated cleaned_corpus.csv.")

filename, contents = next(iter(uploaded.items()))

df = pd.read_csv(
    io.BytesIO(contents),
    keep_default_na=False,
)

required = {
    "id", "platform", "text_for_analysis",
    "cleaned_text", "is_synthetic",
}
missing = required - set(df.columns)

if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

assert len(df) == 360, f"Expected 360 rows, found {len(df)}."
assert df["id"].is_unique
assert df["cleaned_text"].str.strip().ne("").all()
assert df["is_synthetic"].astype(str).str.lower().eq("true").all()

print("Expanded dataset loaded.")
print(df.groupby("platform").size())
print("Distinct cleaned texts:", df["cleaned_text"].nunique())

Saving cleaned_corpus.csv to cleaned_corpus (1).csv
Expanded dataset loaded.
platform
linkedin    180
reddit      180
dtype: int64
Distinct cleaned texts: 360


In [8]:
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# Preserve row order so topic assignments match the correct records.
df = df.reset_index(drop=True)
documents = df["cleaned_text"].tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)

embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

assert embeddings.shape[0] == len(df)
assert np.isfinite(embeddings).all()

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

# Stopwords are removed only when extracting topic keywords.
# Your stored text columns remain unchanged.
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=30,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(
    documents,
    embeddings=embeddings,
)

df["topic_id"] = topics

print("\nTraining finished.")
print("Records assigned:", len(df))
print("Topics found:", df.loc[df["topic_id"] != -1, "topic_id"].nunique())
print("Outlier records:", int(df["topic_id"].eq(-1).sum()))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2026-10-02 09:04:55,564 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-02 09:05:05,671 - BERTopic - Dimensionality - Completed ✓
2026-10-02 09:05:05,673 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-02 09:05:05,739 - BERTopic - Cluster - Completed ✓
2026-10-02 09:05:05,748 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-02 09:05:05,810 - BERTopic - Representation - Completed ✓



Training finished.
Records assigned: 360
Topics found: 3
Outlier records: 131


In [9]:
topic_info = topic_model.get_topic_info()

display(
    topic_info[["Topic", "Count", "Name", "Representation"]]
)

platform_counts = pd.crosstab(
    df["topic_id"],
    df["platform"],
).reindex(columns=["reddit", "linkedin"], fill_value=0)

display(platform_counts)

outlier_rate = df["topic_id"].eq(-1).mean() * 100
print(f"Outlier percentage: {outlier_rate:.1f}%")

,Topic,Count,Name,Representation
0,-1,131,-1_labor_practice_work_service_company_feel,"[labor_practice, work, service_company, feel, ..."
1,0,133,0_work_role_service_company_feel,"[work, role, service_company, feel, worry, exp..."
2,1,60,1_year_changed_service_company_work,"[year, changed, service_company, work, experie..."
3,2,36,2_office_new_service_company_work,"[office, new, service_company, work, experienc..."


platform,reddit,linkedin
topic_id,,
-1,65,66
0,67,66
1,30,30
2,18,18


Outlier percentage: 36.4%


In [10]:
for topic_id in sorted(df["topic_id"].unique()):
    topic_rows = df[df["topic_id"] == topic_id]

    print("\n" + "=" * 70)
    print(f"TOPIC {topic_id} | Records: {len(topic_rows)}")

    keywords = topic_model.get_topic(int(topic_id)) or []
    print("Keywords:", ", ".join(word for word, _ in keywords[:10]))

    for platform in ("reddit", "linkedin"):
        examples = topic_rows[
            topic_rows["platform"] == platform
        ]["text_for_analysis"]

        print(f"\n{platform.upper()} examples:")

        if examples.empty:
            print("No records from this platform.")
        else:
            for text in examples.sample(
                n=min(2, len(examples)),
                random_state=42,
            ):
                print("-", text)


TOPIC -1 | Records: 131
Keywords: labor_practice, work, service_company, feel, experience, job, employer, team, colleagues supported, colleagues

REDDIT examples:
- I recently joined Infosys. The performance improvement plan gives me little time to meet its targets. I am grateful for colleagues who have supported me.
- My current employer is Wipro. I need a project allocation before my bench period becomes a problem. This has been a learning journey, although it has been difficult.

LINKEDIN examples:
- I work at TCS. The performance improvement plan gives me little time to meet its targets. I feel frustrated by the lack of clear communication.
- I work at a product startup. I need a project allocation before my bench period becomes a problem. I feel frustrated by the lack of clear communication.

TOPIC 0 | Records: 133
Keywords: work, role, service_company, feel, worry, experience, bench, project, engineering, software

REDDIT examples:
- I recently joined Infosys. Despite positive f

In [11]:
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# Preserve the first result for comparison.
baseline_model = topic_model
baseline_df = df.copy()

# Confirm that the embeddings still correspond to these rows.
assert documents == df["cleaned_text"].tolist()
assert embeddings.shape[0] == len(df)

candidate_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=30,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    ),
    vectorizer_model=CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
    ),
    calculate_probabilities=False,
    verbose=True,
)

candidate_topics, _ = candidate_model.fit_transform(
    documents,
    embeddings=embeddings,
)

candidate_df = df.copy()
candidate_df["topic_id"] = candidate_topics

comparison = pd.DataFrame([
    {
        "run": "Original",
        "topics": baseline_df.loc[
            baseline_df["topic_id"] != -1, "topic_id"
        ].nunique(),
        "outliers": int(baseline_df["topic_id"].eq(-1).sum()),
        "outlier_percent": round(
            baseline_df["topic_id"].eq(-1).mean() * 100, 1
        ),
    },
    {
        "run": "min_samples=5",
        "topics": candidate_df.loc[
            candidate_df["topic_id"] != -1, "topic_id"
        ].nunique(),
        "outliers": int(candidate_df["topic_id"].eq(-1).sum()),
        "outlier_percent": round(
            candidate_df["topic_id"].eq(-1).mean() * 100, 1
        ),
    },
])

display(comparison)

2026-10-02 10:19:06,368 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-02 10:19:07,198 - BERTopic - Dimensionality - Completed ✓
2026-10-02 10:19:07,200 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-02 10:19:07,220 - BERTopic - Cluster - Completed ✓
2026-10-02 10:19:07,228 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-02 10:19:07,261 - BERTopic - Representation - Completed ✓


,run,topics,outliers,outlier_percent
0,Original,3,131,36.4
1,min_samples=5,3,108,30.0


In [12]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from bertopic.vectorizers import ClassTfidfTransformer

keyword_stopwords = sorted(
    set(ENGLISH_STOP_WORDS)
    | {
        "service_company",
        "product_company",
        "labor_practice",
    }
)

candidate_model.update_topics(
    documents,
    vectorizer_model=CountVectorizer(
        stop_words=keyword_stopwords,
        ngram_range=(1, 2),
        min_df=2,
    ),
    ctfidf_model=ClassTfidfTransformer(
        reduce_frequent_words=True
    ),
)

display(
    candidate_model.get_topic_info()[
        ["Topic", "Count", "Name", "Representation"]
    ]
)

,Topic,Count,Name,Representation
0,-1,108,-1_job_employer_work_team,"[job, employer, work, team, experience, despit..."
1,0,156,0_role_feel_work_project,"[role, feel, work, project, bench, engineering..."
2,1,60,1_changed_year_experience_work,"[changed, year, experience, work, colleagues, ..."
3,2,36,2_office_new_experience_work,"[office, new, experience, work, communication ..."


In [13]:
for topic_id in sorted(candidate_df["topic_id"].unique()):
    rows = candidate_df[
        candidate_df["topic_id"] == topic_id
    ]

    print(f"\nTOPIC {topic_id} — {len(rows)} records")

    keywords = candidate_model.get_topic(int(topic_id)) or []
    print("Keywords:", ", ".join(word for word, _ in keywords[:10]))

    examples = candidate_model.get_representative_docs(
        int(topic_id)
    ) or []

    print("Representative normalized texts:")
    for text in examples[:3]:
        print("-", text)

    print("Additional analysis-text examples:")
    for text in rows["text_for_analysis"].sample(
        n=min(3, len(rows)),
        random_state=42,
    ):
        print("-", text)


TOPIC -1 — 108 records
Keywords: job, employer, work, team, experience, despite, announced, new, communication, hopeful
Representative normalized texts:
- I work for [SERVICE_COMPANY]. My [LABOR_PRACTICE] notice_period conflicts with the new employer's joining date. I am documenting the experience to understand my next steps.
- I recently joined [SERVICE_COMPANY]. My [LABOR_PRACTICE] notice_period conflicts with the new employer's joining date. I feel frustrated by the lack of clear communication.
- I work for [SERVICE_COMPANY]. Negotiating a [LABOR_PRACTICE] notice_period buyout has delayed my job switch. I feel frustrated by the lack of clear communication.
Additional analysis-text examples:
- I work on cloud infrastructure. My next job offer may expire before I finish serving notice. I feel frustrated by the lack of clear communication.
- I recently moved into a testing role. My 90-day notice period conflicts with the new employer's joining date. I am grateful for colleagues who ha

In [14]:
import pandas as pd

topic_model = candidate_model
final_df = candidate_df.copy()

topic_labels = {
    -1: "Outlier / Unassigned",
    0: "Mixed Employment and Workplace Concerns",
    1: "Compensation and Appraisals",
    2: "Office Attendance and Commuting",
}

actual_ids = set(final_df["topic_id"].astype(int))

assert actual_ids == set(topic_labels), (
    "Topic IDs differ from the reviewed result. Review labels first."
)
assert len(final_df) == 360
assert final_df["id"].is_unique
assert final_df["topic_id"].tolist() == list(topic_model.topics_)

final_df["topic_name"] = final_df["topic_id"].map(topic_labels)
final_df["model_version"] = "synthetic-demo-v1"

assert final_df["topic_name"].notna().all()

topic_model.set_topic_labels(topic_labels)

display(
    final_df.groupby(
        ["topic_id", "topic_name", "platform"]
    ).size().unstack(fill_value=0)
)

,platform,linkedin,reddit
topic_id,topic_name,,
-1,Outlier / Unassigned,54,54
0,Mixed Employment and Workplace Concerns,78,78
1,Compensation and Appraisals,30,30
2,Office Attendance and Commuting,18,18


In [15]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

release_name = (
    "AnxioMeter_Phase1_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
release_dir = Path("/content") / release_name

for folder in ("data", "models", "reports"):
    (release_dir / folder).mkdir(parents=True, exist_ok=True)

# Main input for Developer 2.
dataset_path = release_dir / "data" / "topic_mapped_data.csv"
final_df.to_csv(dataset_path, index=False, encoding="utf-8")

# Explicit mapping of topic IDs to names.
labels_df = pd.DataFrame([
    {"topic_id": topic_id, "topic_name": name}
    for topic_id, name in sorted(topic_labels.items())
])
labels_df.to_csv(
    release_dir / "data" / "topic_labels.csv",
    index=False,
)

# Model suitable for compact inference.
model_path = release_dir / "models" / "bertopic_model"
topic_model.save(
    str(model_path),
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="sentence-transformers/all-MiniLM-L6-v2",
)

# Review material.
topic_model.get_topic_info().to_csv(
    release_dir / "reports" / "topic_summary.csv",
    index=False,
)

pd.crosstab(
    final_df["topic_id"], final_df["platform"]
).to_csv(
    release_dir / "reports" / "topic_platform_counts.csv"
)

comparison.to_csv(
    release_dir / "reports" / "model_comparison.csv",
    index=False,
)

packages = [
    "bertopic",
    "sentence-transformers",
    "torch",
    "transformers",
    "huggingface-hub",
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "umap-learn",
    "hdbscan",
    "safetensors",
]
package_versions = {name: version(name) for name in packages}

metadata = {
    "model_version": "synthetic-demo-v1",
    "data_type": "synthetic",
    "records": len(final_df),
    "platform_counts": {
        name: int(count)
        for name, count in final_df["platform"].value_counts().items()
    },
    "outliers": int(final_df["topic_id"].eq(-1).sum()),
    "outlier_percent": float(
        final_df["topic_id"].eq(-1).mean() * 100
    ),
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_input": "cleaned_text",
    "umap": {
        "n_neighbors": 15,
        "n_components": 5,
        "min_dist": 0.0,
        "metric": "cosine",
        "random_state": 42,
    },
    "hdbscan": {
        "min_cluster_size": 30,
        "min_samples": 5,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "prediction_data": True,
    },
    "serialization": "safetensors",
    "python_version": platform.python_version(),
    "packages": package_versions,
    "dataset_sha256": hashlib.sha256(
        dataset_path.read_bytes()
    ).hexdigest(),
    "limitations": [
        "All records are synthetic.",
        "360 examples derive from 30 core statements.",
        "Topic 0 contains multiple workplace themes.",
        "Labels are provisional, based on inspected examples.",
        "Platform differences are not real-world findings.",
        "Compact model does not preserve original UMAP/HDBSCAN objects.",
    ],
}

(release_dir / "reports" / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

(release_dir / "requirements-modeling.txt").write_text(
    "\n".join(
        f"{name}=={package_versions[name]}"
        for name in packages
    ) + "\n",
    encoding="utf-8",
)

handoff = """# AnxioMeter Phase 1 — Synthetic Prototype

## Start here
Load data/topic_mapped_data.csv with pandas.
No scraping or topic retraining is required to begin Phase 2.

## Column usage
- id: stable record identifier; preserve it in downstream outputs.
- platform: synthetic platform assignment, reddit or linkedin.
- created_utc: invented publication timestamp.
- text: original synthetic wording.
- text_for_analysis: input for emotion classification and fluff scoring.
- cleaned_text: normalized input used for topic modeling.
- topic_id: assigned topic; -1 means Outlier / Unassigned.
- topic_name: provisional human-reviewed topic label.
- is_synthetic: must remain attached to downstream records.
- model_version: identifies this dataset/model release.

## Analysis rules
Topic 0 is a mixed workplace-concerns cluster.
Do not rename it to a narrower theme without reviewing its records.
Report outliers separately or state explicitly when excluding them.
Never present these synthetic platform comparisons as research findings.
The samples do not establish Reddit/LinkedIn sentiment differences.

## Model
models/bertopic_model contains a compact safetensors BERTopic model.
The embedding model may download on first use.
Compact inference uses topic embeddings and can produce assignments
different from the original density-based training assignments.
Use the topic IDs already stored in the CSV for this corpus.

## Reproducibility
reports/run_metadata.json records settings and package versions.
requirements-modeling.txt pins the listed modeling packages from Colab;
it is not a complete lock of all transitive dependencies.
Keep the completed notebook and source-generation/cleaning scripts
alongside this release. The notebook captures keyword-extraction settings.

## Phase 2
Add emotion and index columns while preserving record IDs and provenance.
Write the resulting dataset to data/final_analytics.csv.
"""

(release_dir / "HANDOFF.md").write_text(
    handoff,
    encoding="utf-8",
)

print("Exported release:", release_dir)
print("Dataset rows:", len(final_df))
print("Model saved:", model_path.exists())

Exported release: /content/AnxioMeter_Phase1_20261002T102358680589Z
Dataset rows: 360
Model saved: True


In [16]:
from bertopic import BERTopic

saved_df = pd.read_csv(dataset_path, keep_default_na=False)

assert len(saved_df) == 360
assert saved_df["id"].is_unique
assert saved_df["topic_name"].ne("").all()
assert saved_df["is_synthetic"].astype(str).str.lower().eq("true").all()
assert saved_df["topic_id"].tolist() == final_df["topic_id"].tolist()

loaded_model = BERTopic.load(
    str(model_path),
    embedding_model=embedding_model,
)

test_topics, _ = loaded_model.transform(
    final_df["cleaned_text"].head(3).tolist()
)

assert len(test_topics) == 3
assert set(map(int, test_topics)).issubset(set(topic_labels))

print("Dataset verification passed.")
print("Saved model reload passed.")
print("Inference check passed.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-02 10:25:23,755 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Dataset verification passed.
Saved model reload passed.
Inference check passed.


In [17]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    str(release_dir),
    "zip",
    root_dir=str(release_dir),
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>